<a href="https://colab.research.google.com/github/steveonyeke/python-ai-governance/blob/main/phase3-llm-evaluation/03_llm_as_judge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 3: LLM as a Judge
Goal: Use an LLM to automatically score and evaluate LLM responses
Date: May 2026.
Status: In Progress

In [6]:
import time
import pandas as pd
from google import genai
from google.colab import userdata, drive
import os

# Setup
drive.mount('/content/drive')
SAVE_PATH = "/content/drive/MyDrive/python-ai-governance/data/"
os.makedirs(SAVE_PATH, exist_ok=True)
client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))

# Function 1 - Generate a response
def ask_llm(prompt):
  response = client.models.generate_content(
      model="gemini-flash-latest",
      contents=prompt
  )
  return response.text

# Function 2 - Judge a response
def judge_response(prompt, response):
  judge_prompt = f"""
You are an expert AI governance evaluator.
Score the following response on three criteria
Return ONLY a JSON object, nothing else.

Question asked: {prompt}

Response to evaluator: {response}

Score each criterion from 1-5:
- accuracy: Is the information factually correct?
- clarity: Is the response clear and well structured?
- relevance: Does the response directly address the question?

Also provide a one sentence summary of your evaluation.

Return exactly this format:
{{
  "accuracy": <score>,
  "clarity": <score>,
  "relevance": <score>,
  "summary": "<one sentence>"
}}
"""
  result = client.models.generate_content(
      model="gemini-flash-latest",
      contents=judge_prompt
  )
  return result.text

# Test prompts
test_prompts = [
    "What is AI governance in two sentences?",
    "What are the main risks of unregulated AI?",
    "Why does the EU AI Act matter for businesses?",
]

# Run eval and judge
results = []

print("====== LLM AS JUDGE EVALUATION ======\n")

for i, prompt in enumerate(test_prompts):
  print(f"[{i+1}] Generating response...")
  response = ask_llm(prompt)
  time.sleep(2)

  print(f"[{i+1}] Judging response...")
  judgment = judge_response(prompt, response)
  time.sleep(2)

  # Clean up JSON from judge
  import json
  import re
  clean = re.sub(r"'''json/'''", "", judgment).strip()

  try:
    scores = json.loads(clean)
    results.append({
        "prompt_number": i + 1,
        "prompt":        prompt,
        "response":      response,
        "accuracy":      scores["accuracy"],
        "clarity":       scores[:"clarity"],
        "relevance":     scores["relevance"],
        "summary":       scores["summary"],
    })
    print(f"     Accuracy: {scores['accuracy']}/5")
    print(f"   Clarity: {scores['clarity']}/5")
    print(f"Relevance: {scores['relevance']}/5")
    print(f"  Summary: {scores['summary']}\n")

  except:
    print(f"""     Could not parse judgment - raw output:
  {judgment}
""")

# Build dataframe
df = pd.Dataframe(results)

print("====== JUDGMENT SUMMARY ======")
print(f"Avg accuracy: {df['accuracy'].mean():.1f}/5")
print(f"Avg clarity: {df['clarity'].mean():.1f}/5")
print(f"Avg relevance: {df['relevance'].mean():.}/5")

# Save
df.to_csv(SAVE_PATH + "llm_as_judge_results.csv", index=False)
print(f"Results saved to {SAVE_PATH}llm_as_judge_results.csv ✅")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
====== LLM AS JUDGE EVALUATION ======

[1] Generating response...


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3-flash\nPlease retry in 27.794598155s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3-flash'}, 'qu